# 手写数字分类：逐步复现

我用这个 Notebook 学习数据 → 选参 → 训练 → 测试 → 错误分析的流程。

从上到下运行。先安装 `requirements-notebook.txt`，选择仓库 `.venv` 内核。这里调用 `digits_workflow.py` 中的函数，和脚本共用同一实验实现。

已保存输出来自助手验证执行；我的理解与修改记录在 `LEARNING_TASKS.md`，末尾留有实际作答区域。


## 1. 环境与函数

`import` 导入已有工具。`Path` 管理文件位置；后面结果会保存到仓库 `results/`，覆盖原有生成文件。这里检查当前工作目录能否找到实验函数。


In [1]:
from pathlib import Path
import csv
import numpy as np
from IPython.display import Image, display

assert Path("digits_workflow.py").exists(), "请在仓库目录打开 Notebook"
from digits_workflow import load_split, tune_knn, make_models, evaluate_models, save_run_info

output = Path("results")
output.mkdir(exist_ok=True)
print("实验函数已导入，结果目录：results")


实验函数已导入，结果目录：results


## 2. 读取数据并看形状

`X` 是图片的特征，`y` 是正确标签；`train` 和 `test` 是划分后的样本编号。

先预测三个 shape 会打印什么，再运行核对。`reshape` 把一张 8×8 图展平成 64 个数，不改变像素内容。


In [2]:
digits, X, y, train, test = load_split()
print("images:", digits.images.shape)
print("X:", X.shape)
print("y:", y.shape)
print("训练 / 测试数量:", len(train), len(test))
print("第一张图的标签:", y[0])
print("第一张图的灰度矩阵:\n", digits.images[0])


images: (1797, 8, 8)
X: (1797, 64)
y: (1797,)
训练 / 测试数量: 1347 450
第一张图的标签: 0
第一张图的灰度矩阵:
 [[ 0.  0.  5. 13.  9.  1.  0.  0.]
 [ 0.  0. 13. 15. 10. 15.  5.  0.]
 [ 0.  3. 15.  2.  0. 11.  8.  0.]
 [ 0.  4. 12.  0.  0.  8.  8.  0.]
 [ 0.  5.  8.  0.  0.  9.  8.  0.]
 [ 0.  4. 11.  0.  1. 12.  7.  0.]
 [ 0.  2. 14.  5. 10. 12.  0.  0.]
 [ 0.  0.  6. 13. 10.  0.  0.  0.]]


## 3. 核查划分

训练集提供学习材料，测试集用于最后评分。两部分没有重叠，且每种数字都出现在两部分中。`stratify=y` 用于尽量保持类别比例。


In [3]:
assert set(train).isdisjoint(test)
print("label | train | test")
for label in range(10):
    print(label, int(np.sum(y[train] == label)), int(np.sum(y[test] == label)))


label | train | test
0 133 45
1 136 46
2 133 44
3 137 46
4 136 45
5 136 46
6 136 45
7 134 45
8 131 43
9 135 45


## 4. 仅用训练集选择 KNN 的 k

函数内对 k=1、3、5、7、9 做五折交叉验证，标准化器也在每折内部拟合。调用时只传入训练部分；不传入测试图片或答案。

打印的是验证折准确率的平均值，不能当作测试分数。完整五折数值保存为 `knn_cv.csv`。


In [4]:
best_k, cv_rows = tune_knn(X[train], y[train], output)
for row in cv_rows:
    print(f"k={row['k']}: CV mean={row['mean_cv_accuracy']:.4f}, std={row['std_cv_accuracy']:.4f}")
print("选中的 k:", best_k)
display(Image(filename=str(output / "knn_cv.png")))


k=1: CV mean=0.9681, std=0.0076
k=3: CV mean=0.9733, std=0.0043
k=5: CV mean=0.9748, std=0.0054
k=7: CV mean=0.9666, std=0.0066
k=9: CV mean=0.9636, std=0.0072
选中的 k: 5


![训练集五折交叉验证](results/knn_cv.png)

## 5. 训练并测试

函数内部对每个模型执行 `fit(X[train], y[train])`，再执行 `predict(X[test])`。比较预测与真实标签，得到准确率和错误数。

固定 SVM、固定 k=5 的 KNN 是原始基线；第三项 KNN 使用上一步的选择。若选出的 k=5，两个 KNN 本来就是相同设置，不应宣称调参提高了成绩。


In [5]:
models = make_models(best_k)
metrics = evaluate_models(models, digits, X, y, train, test, output)
save_run_info(output, train, test, best_k, cv_rows)
for row in metrics:
    print(f"{row['model']}: {row['correct']}/{row['test_samples']} correct, "
          f"accuracy={row['accuracy']:.2%}, errors={row['errors']}")


svm: 441/450 correct, accuracy=98.00%, errors=9
knn: 434/450 correct, accuracy=96.44%, errors=16
knn_tuned: 434/450 correct, accuracy=96.44%, errors=16


## 6. 从混淆矩阵找到错误

横轴是预测标签，纵轴是真实标签；对角线上是正确预测。例如真实 8、预测 1，会落在行标签 8、列标签 1 的位置。

再看错误图片。`id` 是原始数据编号，`true` 是真实标签，`pred` 是预测标签。这里只展示前八个错误；全部错误在 `errors.csv`。


In [6]:
display(Image(filename=str(output / "svm_confusion_matrix.png")))
display(Image(filename=str(output / "svm_errors.png")))


![SVM 混淆矩阵](results/svm_confusion_matrix.png)

![SVM 前八个错误](results/svm_errors.png)

## 7. 定位一张错误图片

下面取 SVM 的第一条错误记录。`dataset_index` 可以定位原图，`test_position` 可以定位测试数组中的位置；两者不能混用。

练习：把索引 `0` 改成 `1` 或 `2`，观察另一个错误。不要直接修改 CSV 中的预测。


In [7]:
with (output / "errors.csv").open(encoding="utf-8", newline="") as handle:
    errors = list(csv.DictReader(handle))
svm_errors = [row for row in errors if row["model"] == "svm"]
sample = svm_errors[0]
idx = int(sample["dataset_index"])
print("错误记录:", sample)
print("原始标签核查:", y[idx])
print("对应的 8×8 像素矩阵:\n", digits.images[idx])


错误记录: {'model': 'svm', 'test_position': '25', 'dataset_index': '129', 'true_label': '8', 'predicted_label': '1', 'is_error': '1'}
原始标签核查: 8
对应的 8×8 像素矩阵:
 [[ 0.  0.  7. 13. 15.  5.  0.  0.]
 [ 0.  0.  8. 16. 16. 12.  0.  0.]
 [ 0.  0.  7. 16. 15.  3.  0.  0.]
 [ 0.  0.  6. 16.  5.  0.  0.  0.]
 [ 0.  0.  5. 16.  2.  0.  0.  0.]
 [ 0.  0.  8. 16.  6.  0.  0.  0.]
 [ 0.  0. 12. 12. 13.  0.  0.  0.]
 [ 0.  0.  5. 13. 10.  0.  0.  0.]]


## 8. 检查结果是否一致

检查划分无重叠、保存的正确/错误数与逐条预测一致、k 选择与 CV 表一致，并核查训练统计和 SVM 预测。此检查不证明模型在其他数据上有效。


In [8]:
from verify_results import verify
verify(output)


PASS: split, training-only scaling, CV selection, predictions and metrics are consistent.
SVM error pairs: {('8', '1'): 2, ('9', '7'): 1, ('5', '9'): 1, ('4', '7'): 1, ('7', '5'): 1, ('1', '4'): 1, ('8', '4'): 1, ('9', '6'): 1}


## 9. 我的理解与实际记录（完成后填写）

- 我实际运行的日期：待填写
- 8×8 如何变成 64 个特征：待填写
- fit 和 predict 的区别：待填写
- 为什么不能用测试集选 k：待填写
- 我查看的三个错误编号、标签与观察：待填写
- 我修改的代码与实际输出：待填写
- 我参考或使用 AI 的部分：待填写
- 我仍不能解释的内容：待填写

## 10. 我的独立练习

统计 SVM 每个真实数字被预测错的数量，输出十行。总和应等于该模型的错误数。下面留空给我自己写；不需要重新训练模型。


In [9]:
# 在这里写我自己的统计代码
